
# General Denoise —  UI 
Designed and developed by ASK (A. Shanmukha Kiran); Contact: askiran@alumni.iitm.ac.in

In [2]:
import inspect, threading, traceback
import ipywidgets as W
from IPython.display import display
from pathlib import Path
from general_denoise.common import naming as N


# ---- your package imports ----
import general_denoise.pipeline as P
from general_denoise.common.config import DatasetConfig
from general_denoise.common import backend as B, io
from general_denoise.common.visualize import show_pair

apply_sequence = P.apply_sequence
process_slab   = P.process_slab
run_on_folder  = P.run_on_folder
ALGO_REGISTRY  = P.ALGO_REGISTRY               # {"median3d": fn.apply, ...}
AVAIL          = set(ALGO_REGISTRY.keys())     # Available algorithm keys

# ------------------ friendly display names ------------------
DISPLAY_NAMES = {
    "median3d":"Median","gaussian3d":"Gaussian","unsharp3d":"Unsharp",
    "log3d":"Laplacian of Gaussian","fft3d":"FFT","bilateral3d":"Bilateral",
    "nlm3d":"Non-local means","cdor3d":"Contrast-Dependent Outlier Removal",
    "anisotropicdiff3d":"Anisotropic Diffusion","midrange3d":"Midrange",
    "wiener3d":"Wiener","guided3d":"Guided","kalman3d":"Kalman","knn3d":"K-NN", "clahe3d": "CLAHE",
    "mean3d":  "Mean",
}
def _disp(name): return DISPLAY_NAMES.get(name, name)

# ------------------ UI heuristics / overrides ------------------
# Dropdown choices per filter
CHOICES = {
    "fft3d": {"mode":["3d","2d"], "filter_kind":["ideal","butter","gauss"], "pass_type":["low","high","band"]},
    "cdor3d": {"sigma_mode":["mad","std"], "contrast":["range","std","grad"], "replace":["soft","median"]},
    "anisotropicdiff3d": {"func":["exp","lorentz"]},
    "knn3d": {"metric":["l1","l2"], "weighting":["inverse","inverse_sq","gaussian","uniform"]},
    "nlm3d": {"use_box":[True, False]},
    "clahe3d": {"z_mode": ["per_slice"], "method": ["auto","skimage"]},
}

# Parameters that should live in Advanced pane
ADV_KEYS = {
    "fft3d": {"spacing","use_nyquist","f_lo_nyq","f_hi_nyq","f_lo_abs","f_hi_abs","shift"},
    "unsharp3d": {"clip_min","clip_max"},
    "nlm3d": {"sigma","use_box"},
    "cdor3d": {"sigma_mode","contrast","replace"},
    "anisotropicdiff3d": {"func","vox"},
    "knn3d": {"w_eps","chunk","tile"},
    "wiener3d": {"sigma_n"},
    "clahe3d": {"nbins","tile","z_mode","method"},
}

# Tuple kernel/window names that must be odd-only
ODD_KEYS = {"size","win"}

# Log sliders
LOG_KEYS = {"eps","w_eps"}

# Default float ranges for sliders; fallbacks built heuristically
RANGE = {
    "amount": (0.0, 3.0, 0.05),
    "threshold": (0.0, 0.5, 0.01),
    "k": (0.005, 0.25, 0.005),
    "lam": (0.01, 0.30, 0.01),
    "sigma_r": (0.01, 0.5, 0.01),
    "w_sigma": (0.01, 1.0, 0.01),
    "f_lo_nyq": (0.0, 1.0, 0.01),
    "f_hi_nyq": (0.0, 1.0, 0.01),
    "R": (1e-4, 0.5, 0.005),
    "Q": (1e-6, 1e-2, 1e-6),
    "init_var": (0.1, 10.0, 0.1),
}

# Optional presets (per filter)
PRESETS = {
    "median3d": {"Default":{"size":(3,3,3)},"Strong":{"size":(21,21,21)}},
    "gaussian3d": {"Default":{"sigma":(1,1,1)},"Soft":{"sigma":(0.7,0.7,0.7)},"Strong":{"sigma":(5,5,5)}},
    "unsharp3d": {"Default":{"sigma":(1,1,1),"amount":0.6,"threshold":0.0},
                  "Punchy":{"sigma":(1.5,1.5,1.5),"amount":1.5,"threshold":0.0}},
    "fft3d": {"Default":{"mode":"3d","filter_kind":"butter","pass_type":"low","order":2,"use_nyquist":True,"f_hi_nyq":0.25,"shift":False},
              "HighPass":{"mode":"3d","filter_kind":"butter","pass_type":"high","order":2,"use_nyquist":True,"f_lo_nyq":0.05}},
    "cdor3d": {"Default":{"win":(3,5,5),"k_sigma":3.0,"alpha":0.4}},
    "anisotropicdiff3d": {"Default":{"iters":50,"k":0.05,"lam":0.15,"func":"exp"}},
    "clahe3d": {"Default":  {"clip_limit": 0.01, "tile": (8,8), "method":"auto"},"Stronger": {"clip_limit": 0.02, "tile": (8,8), "method":"auto"}},
    "mean3d": {"Default": {"size": (3,3,3)},"Strong":  {"size": (7,7,7)}},
}

# ------------------ dataset panel (left) ------------------
in_folder   = W.Text(value="/home/askiran/data/Normalized_log/", description="in_folder", layout=W.Layout(width="560px"))
out_folder  = W.Text(value="/home/askiran/data/Normalized_log/denoised/", description="out_folder", layout=W.Layout(width="560px"))
pattern     = W.Text(value="*.[tT][iI][fF]*", description="pattern", layout=W.Layout(width="260px"))
dtype_in    = W.Dropdown(options=["uint8","uint16","float32","float64"], value="uint8", description="dtype")
dtype_out   = W.Dropdown(options=["uint8","uint16","float32","float64"], value="uint8", description="out_dtype")
z_overlap   = W.BoundedIntText(value=2, min=0, max=9999, description="z_overlap")

compression = W.Dropdown(options=[None, "zlib", "zstd", "lzma", "jpeg"], value=None, description="compression")
mem_frac    = W.FloatSlider(value=0.6, min=0.3, max=0.9, step=0.05, readout_format=".2f", description="mem_frac")
min_slab_z  = W.BoundedIntText(value=1, min=1, max=9999, description="min_slab_z")
slab_auto   = W.Checkbox(value=True, description="slab_z auto")
slab_z      = W.BoundedIntText(value=96, min=1, max=99999, description="slab_z", disabled=True)
def _toggle_slab(ch): slab_z.disabled = ch["new"]
slab_auto.observe(_toggle_slab, names="value")

dataset_panel = W.VBox([
    W.HTML("<b>Dataset</b>"),
    in_folder, out_folder,
    W.HBox([pattern, dtype_in, dtype_out]),
    W.HBox([z_overlap, compression]),
    W.HBox([W.Label("Memory:"), mem_frac, min_slab_z]),
    W.HBox([slab_auto, slab_z]),
], layout=W.Layout(border="1px solid #444", padding="6px"))

# ------------------ widget helpers ------------------
def _IntOddSlider(value, description=""):
    v = value + (value % 2 == 0)
    return W.IntSlider(value=v, min=1, max=99, step=2, readout=True,
                       description=description, layout=W.Layout(width="260px"))

def _IntSlider(value, description="", lo=1, hi=4096, step=1):
    return W.IntSlider(value=int(value), min=lo, max=hi, step=step, readout=True,
                       description=description, layout=W.Layout(width="260px"))

def _float_range(name, default):
    if name in RANGE: return RANGE[name]
    if default == 0:  return (0.0, 1.0, 0.01)
    mag = abs(float(default))
    return (0.0, max(3*mag, 1.0), max(0.01*mag, 0.001))

def _FloatSlider(value, name):
    lo, hi, st = _float_range(name, value)
    return W.FloatSlider(value=float(value), min=lo, max=hi, step=st, readout_format=".4f",
                         description=name, layout=W.Layout(width="320px"))

def _FloatLogSlider(value, description=""):
    return W.FloatLogSlider(value=float(value), base=10, min=-6, max=-2, step=0.1,
                            description=description, layout=W.Layout(width="320px"))

def _tuple_kind(name, tval):
    if len(tval) == 3 and all(isinstance(x, (int, float)) for x in tval):
        if name in ODD_KEYS and all(isinstance(x, int) for x in tval):
            return "int3_odd"
        return "float3" if any(isinstance(x, float) for x in tval) else "int3"
    if len(tval) == 2 and all(isinstance(x, int) for x in tval):
        return "int2"
    return "tuple"

def _is_advanced(filter_name, pname): return pname in ADV_KEYS.get(filter_name, set())

# ------------------ INTROSPECTIVE PARAM FORM ------------------
def build_param_form(filter_name):
    """Build a fresh (Accordion, controls_dict) for the chosen filter."""
    fn = ALGO_REGISTRY[filter_name]        # this *is* the apply() function
    sig = inspect.signature(fn)

    controls = {}
    basic_rows, adv_rows = [], []

    # (optional) presets
    preset_dd = None
    if filter_name in PRESETS:
        preset_dd = W.Dropdown(options=sorted(PRESETS[filter_name].keys()),
                               value=sorted(PRESETS[filter_name].keys())[0],
                               description="Preset")
        basic_rows.append(preset_dd)

    for p in sig.parameters.values():
        name = p.name
        if name in ("vol", "iterations", "kwargs", "args"):
            continue
        if p.kind in (p.VAR_KEYWORD, p.VAR_POSITIONAL):
            continue
        default = None if (p.default is inspect._empty) else p.default

        # Choice override (Dropdown)
        if filter_name in CHOICES and name in CHOICES[filter_name]:
            opts = CHOICES[filter_name][name]
            val = default if default in opts else (opts[0] if opts else default)
            w = W.Dropdown(options=opts, value=val, description=name, layout=W.Layout(width="300px"))
            controls[name] = w
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(w)
            continue

        # Tuple params
        if isinstance(default, tuple):
            kind = _tuple_kind(name, default)
            if kind == "float3":
                z = _FloatSlider(default[0], name); y = _FloatSlider(default[1], name); x = _FloatSlider(default[2], name)
                controls[name] = (z,y,x)
                row = W.VBox([W.Label(f"{name} (Z,Y,X)"), z,y,x])
            elif kind == "int3_odd":
                z = _IntOddSlider(default[0], "Z"); y = _IntOddSlider(default[1], "Y"); x = _IntOddSlider(default[2], "X")
                controls[name] = (z,y,x)
                row = W.VBox([W.Label(f"{name} (Z,Y,X)"), z,y,x])
            elif kind == "int3":
                z = _IntSlider(default[0], "Z"); y = _IntSlider(default[1], "Y"); x = _IntSlider(default[2], "X")
                controls[name] = (z,y,x)
                row = W.VBox([W.Label(f"{name} (Z,Y,X)"), z,y,x])
            elif kind == "int2":
                ty = _IntSlider(default[0], "Ty"); tx = _IntSlider(default[1], "Tx")
                controls[name] = (ty, tx)
                row = W.VBox([W.Label(f"{name} (Ty,Tx)"), ty, tx])
            else:
                row = W.Label(f"{name} (tuple) = {default}")
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(row)
            continue

        # Bool
        if isinstance(default, bool):
            w = W.Checkbox(value=default, description=name)
            controls[name] = w
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(w)
            continue

        # None -> optional float control
        if default is None:
            cb = W.Checkbox(value=False, description=f"{name} enabled")
            ft = W.FloatText(value=0.0, description=name, disabled=True, layout=W.Layout(width="220px"))
            def _toggle(ch, ft=ft): ft.disabled = not ch["new"]
            cb.observe(_toggle, names="value")
            controls[name] = (cb, ft)
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(W.HBox([cb, ft]))
            continue

        # Int
        if isinstance(default, int):
            w = _IntSlider(default, name)
            controls[name] = w
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(w)
            continue

        # Float
        if isinstance(default, float):
            if name in LOG_KEYS:
                w = _FloatLogSlider(default, name)
            else:
                w = _FloatSlider(default, name)
            controls[name] = w
            (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(w)
            continue

        # Fallback
        (adv_rows if _is_advanced(filter_name, name) else basic_rows).append(W.Label(f"{name} = {default!r}"))

    # FFT Nyquist/Abs toggle wiring
    if filter_name == "fft3d" and "use_nyquist" in controls:
        use_nyq = controls["use_nyquist"]
        f_lo_nyq = controls.get("f_lo_nyq"); f_hi_nyq = controls.get("f_hi_nyq")
        f_lo_abs = controls.get("f_lo_abs"); f_hi_abs = controls.get("f_hi_abs")
        def _sync(*_):
            nyq = bool(use_nyq.value)
            for pair in (f_lo_nyq, f_hi_nyq):
                if isinstance(pair, tuple) and pair[0] is not None: pair[0].value = nyq
                if isinstance(pair, tuple) and pair[1] is not None: pair[1].disabled = not nyq
            for pair in (f_lo_abs, f_hi_abs):
                if isinstance(pair, tuple) and pair[0] is not None: pair[0].value = (not nyq)
                if isinstance(pair, tuple) and pair[1] is not None: pair[1].disabled = nyq
        _sync(); use_nyq.observe(_sync, names="value")

    # Presets apply (after controls exist)
    if filter_name in PRESETS and preset_dd is not None:
        def _apply_preset(ch):
            preset = PRESETS[filter_name][ch["new"]]
            for k, v in preset.items():
                if k not in controls: continue
                c = controls[k]
                if isinstance(c, tuple) and len(c)==3 and all(hasattr(x,"value") for x in c):
                    c[0].value, c[1].value, c[2].value = v
                elif isinstance(c, tuple) and len(c)==2 and isinstance(c[0], W.Checkbox):
                    cb, w = c
                    if v is None: cb.value=False; w.disabled=True
                    else: cb.value=True; w.disabled=False; w.value=float(v)
                elif hasattr(c, "value"):
                    c.value = v
        preset_dd.observe(_apply_preset, names="value")

    # Accordion (Advanced opened by default so you see it)
    basic_box = W.VBox(basic_rows or [W.Label("No basic params")])
    adv_box   = W.VBox(adv_rows   or [W.Label("No advanced params")])
    acc = W.Accordion(children=[basic_box, adv_box])
    acc.set_title(0, "Basic"); acc.set_title(1, "Advanced")
    acc.selected_index = 1
    return acc, controls

def read_params(controls):
    params = {}
    for k, c in controls.items():
        if isinstance(c, W.Widget):
            params[k] = (bool(c.value) if isinstance(c, W.Checkbox) else c.value)
        elif isinstance(c, tuple):
            if len(c)==3 and all(hasattr(x,"value") for x in c):
                params[k] = (c[0].value, c[1].value, c[2].value)
            elif len(c)==2 and isinstance(c[0], W.Checkbox):
                cb, w = c; params[k] = (None if not cb.value else float(w.value))
            elif len(c)==2 and all(hasattr(x,"value") for x in c):
                params[k] = (c[0].value, c[1].value)
        else:
            params[k] = c
    return params

# ------------------ sequence editor (left) ------------------
seq = []  # list of {"name":..., "params":..., "enabled": bool}
seq_list = W.Select(options=[], rows=12, layout=W.Layout(width="560px"))
def _label(i, s):
    p = {k:v for k,v in s["params"].items() if k!="iterations"}
    pshort = ", ".join([f"{k}={p[k]}" for k in sorted(p)])
    en = "✓" if s.get("enabled", True) else "✗"
    return f"{i+1:02d}. {en} {_disp(s['name'])} | iter={s['params'].get('iterations',1)} | {pshort}"
def refresh_seq_list(): seq_list.options = [(_label(i,s), i) for i,s in enumerate(seq)]

add_btn=W.Button(description="Add", button_style="success")
update_btn=W.Button(description="Update")
remove_btn=W.Button(description="Remove", button_style="danger")
up_btn=W.Button(description="▲"); down_btn=W.Button(description="▼")
toggle_btn=W.Button(description="Enable/Disable"); clear_btn=W.Button(description="Clear All", button_style="warning")

# ------------------ filter picker + param holder ------------------
filter_pick = W.Dropdown(options=[(_disp(k), k) for k in sorted(AVAIL)],
                         value=("median3d" if "median3d" in AVAIL else sorted(AVAIL)[0]),
                         description="Filter")
iter_box = W.IntSlider(value=1, min=1, max=99, step=1, description="iterations")

# Initial form
param_area, param_controls = build_param_form(filter_pick.value)

# A holder that we fully replace on filter change
param_holder = W.VBox([
    W.HBox([W.Label("Filter:"), filter_pick]),
    W.HBox([W.Label("Iterations:"), iter_box]),
    param_area
])

def _rebuild_param_pane(new_filter):
    global param_area, param_controls
    new_area, new_ctrls = build_param_form(new_filter)
    param_area, param_controls = new_area, new_ctrls
    # Replace children to avoid stale widgets
    param_holder.children = (
        W.HBox([W.Label("Filter:"), filter_pick]),
        W.HBox([W.Label("Iterations:"), iter_box]),
        param_area
    )

def on_filter_change(ch): _rebuild_param_pane(ch["new"])
filter_pick.observe(on_filter_change, names="value")

def _collect_step():
    params = read_params(param_controls)
    params["iterations"] = int(iter_box.value)
    return {"name": filter_pick.value, "params": params, "enabled": True}

def add_step(_): seq.append(_collect_step()); refresh_seq_list()
def update_step(_):
    if seq_list.value is None: return
    seq[int(seq_list.value)] = _collect_step(); refresh_seq_list()
def remove_step(_):
    if seq_list.value is None: return
    seq.pop(int(seq_list.value)); refresh_seq_list()
def move_up(_):
    if seq_list.value is None: return
    i=int(seq_list.value)
    if i>0: seq[i-1],seq[i]=seq[i],seq[i-1]; refresh_seq_list(); seq_list.value=i-1
def move_down(_):
    if seq_list.value is None: return
    i=int(seq_list.value)
    if i < len(seq)-1: seq[i+1],seq[i]=seq[i],seq[i+1]; refresh_seq_list(); seq_list.value=i+1
def toggle_enable(_):
    if seq_list.value is None: return
    i=int(seq_list.value); seq[i]["enabled"]=not seq[i].get("enabled",True); refresh_seq_list()
def clear_all(_): seq.clear(); refresh_seq_list()

for b,fn in [(add_btn,add_step),(update_btn,update_step),(remove_btn,remove_step),
             (up_btn,move_up),(down_btn,move_down),(toggle_btn,toggle_enable),(clear_btn,clear_all)]:
    b.on_click(fn)

left_panel = W.VBox([
    dataset_panel,
    W.HTML("<b>Sequence</b>"),
    seq_list,
    W.HBox([add_btn, update_btn, remove_btn]),
    W.HBox([up_btn, down_btn, toggle_btn, clear_btn]),
])


n_preview = W.IntSlider(value=4, min=1, max=50, step=1, description="n_preview")
z_center  = W.IntSlider(value=0,  min=0, max=0,    step=1, description="z_center")
show_z    = W.IntSlider(value=0,  min=0, max=0,    step=1, description="show_z")
save_prev = W.Checkbox(value=False, description="Save preview")
preview_btn = W.Button(description="Run Preview", button_style="info")
out_prev    = W.Output(layout={"border":"1px solid #ccc"})

def _dconf():
    return DatasetConfig(
        in_folder=in_folder.value, out_folder=out_folder.value,
        pattern=pattern.value or "*.[tT][iI][fF]*",
        dtype=dtype_in.value, out_dtype=dtype_out.value,
        z_overlap=int(z_overlap.value)
    )

def run_preview(_=None):
    out_prev.clear_output()
    with out_prev:
        try:
            seq_effective = [s for s in seq if s.get("enabled", True) and s["name"] in AVAIL]
            if not seq_effective: print("Sequence empty."); return
            dconf  = _dconf()
            paths  = io.list_slices(dconf.in_folder, dconf.pattern)
            Z      = len(paths)
            if Z == 0: print("No input files matched pattern."); return

            n  = int(n_preview.value)
            z0 = max(0, int(z_center.value) - n//2)
            z1 = min(Z, z0 + n)
            if z1 <= z0: z1 = min(Z, z0 + 1)

            host = io.read_slab(paths, z0, z1, dtype=dconf.dtype)
            show_z.max = max(0, host.shape[0]-1)
            if show_z.value > show_z.max: show_z.value = show_z.max

            seq_for_pipeline = [(s["name"], s["params"]) for s in seq_effective]
            y = process_slab(host, dconf.dtype, dconf.out_dtype, seq_for_pipeline)

            if save_prev.value:
                prev_dir = Path(dconf.out_folder) / "preview"
                prev_dir.mkdir(parents=True, exist_ok=True)
                io.write_slab_as_folder(str(prev_dir), y, z_from=z0, compression=compression.value)

                suffix = N.format_sequence_suffix(seq_for_pipeline)
                ok, msg = N.rename_outputs_by_input_order(str(prev_dir), "z_*.tif",
                                                          [str(p) for p in paths[z0:z1]], suffix)
                print("Preview saved →", msg if ok else "Rename skipped -> " + msg)
                print("Suffix:", suffix)

            z_show = int(show_z.value)
            show_pair(host, y, z=z_show, title_a=f"Input (z={z0+z_show})", title_b="Preview")
        except Exception:
            traceback.print_exc()

preview_btn.on_click(run_preview)

preview_controls = W.VBox([n_preview, z_center, show_z, save_prev, preview_btn])
preview_tab = W.HBox([preview_controls, out_prev])

# --- Auto-size n_preview, z_center, show_z from total stack size ---
from pathlib import Path
import ipywidgets as W
from general_denoise.common import io

# small info label under the Dataset panel
try:
    stack_info_lbl
except NameError:
    stack_info_lbl = W.HTML(value="")
try:
    kids = list(dataset_panel.children)
    if stack_info_lbl not in kids:
        kids.append(stack_info_lbl)
        dataset_panel.children = tuple(kids)
except Exception:
    display(stack_info_lbl)

def _window(Z, center, n):
    z0 = max(0, int(center) - int(n)//2)
    z1 = min(int(Z), z0 + int(n))
    if z1 <= z0:
        z1 = min(int(Z), z0 + 1)
    return z0, z1

def _scan_paths():
    try:
        return io.list_slices(in_folder.value, (pattern.value or "*.[tT][iI][fF]*"))
    except Exception:
        return []

def _update_from_stack(*_):
    paths = _scan_paths()
    Z = len(paths)
    stack_info_lbl.value = f"<i>Total slices in folder:</i> <b>{Z}</b>"

    if Z <= 0:
        n_preview.max = 1
        if n_preview.value < 1:
            n_preview.value = 1
        z_center.min = z_center.max = 0
        z_center.value = 0
        show_z.min = show_z.max = 0
        show_z.value = 0
        return

    # n_preview bounded by total Z
    n_preview.max = Z
    if n_preview.value > Z:
        n_preview.value = min(64, Z)

    # z_center at mid-stack by default; keep within [0, Z-1]
    z_center.min = 0
    z_center.max = Z - 1
    if z_center.value == 0:
        z_center.value = Z // 2
    else:
        z_center.value = max(0, min(int(z_center.value), Z - 1))

    # show_z within current preview slab
    z0, z1 = _window(Z, z_center.value, n_preview.value)
    slab_len = max(1, z1 - z0)
    show_z.min = 0
    show_z.max = slab_len - 1
    if show_z.value == 0:
        show_z.value = slab_len // 2
    else:
        show_z.value = max(0, min(int(show_z.value), slab_len - 1))

# reattach observers idempotently
def _safe_unobserve(widget, fn):
    try:
        widget.unobserve(fn, names="value")
    except Exception:
        pass

_safe_unobserve(in_folder, _update_from_stack)
_safe_unobserve(pattern, _update_from_stack)
_safe_unobserve(n_preview, _update_from_stack)
_safe_unobserve(z_center, _update_from_stack)

in_folder.observe(_update_from_stack, names="value")
pattern.observe(_update_from_stack, names="value")
n_preview.observe(_update_from_stack, names="value")
z_center.observe(_update_from_stack, names="value")

# kick once at startup
_update_from_stack()

# ------------------ Full run ------------------
run_btn  = W.Button(description="Run Full (Slabbed)", button_style="primary")
prog     = W.IntProgress(value=0, min=0, max=100, description="Progress")
eta_lbl  = W.Label(value="ETA: --")
msg_lbl  = W.Label(value="")
out_run  = W.Output(layout={"border":"1px solid #ccc"})
diag_btn = W.Button(description="Diagnose input", button_style="warning")
out_diag = W.Output(layout={"border":"1px solid #ccc"})

def diagnose(_):
    out_diag.clear_output()
    with out_diag:
        try:
            dconf = _dconf()
            paths = io.list_slices(dconf.in_folder, dconf.pattern)
            print(f"Found {len(paths)} files.")
            print("First 3:", paths[:3])
            print("Last 3 :", paths[-3:])
        except Exception:
            traceback.print_exc()
diag_btn.on_click(diagnose)

def _progress_cb(current, total, message):
    if prog.max != total: prog.max = int(total)
    prog.value = int(current); msg_lbl.value = message
    try: eta_lbl.value = message.split("|")[-1].strip()
    except Exception: pass

def _do_full(dconf, seq_copy, slab_val):
    try:
        in_paths = io.list_slices(dconf.in_folder, dconf.pattern)
        run_on_folder(dconf, seq_copy, slab_z=slab_val, overwrite=True,
                      progress=_progress_cb, compression=compression.value,
                      mem_frac=float(mem_frac.value), min_slab_z=int(min_slab_z.value))
        suffix = N.format_sequence_suffix(seq_copy)
        ok, msg = N.rename_outputs_by_input_order(dconf.out_folder, "z_*.tif",
                                                  [str(p) for p in in_paths], suffix)
        with out_run:
            print("Post-rename:", msg if ok else "Rename skipped -> " + msg)
            print("Suffix:", suffix)
            print("Done.")
    except Exception:
        with out_run: traceback.print_exc()

def run_full(_):
    out_run.clear_output()
    with out_run:
        print("Starting full-volume run…")
        display(W.HBox([prog, eta_lbl])); display(msg_lbl)
    seq_copy = [(s["name"], s["params"]) for s in seq if s.get("enabled", True) and s["name"] in AVAIL]
    if not seq_copy:
        with out_run: print("No runnable filters."); return
    dconf = _dconf()
    slab_val = None if slab_auto.value else int(slab_z.value)
    prog.value = 0; msg_lbl.value = ""; eta_lbl.value = "ETA: --"
    threading.Thread(target=_do_full, args=(dconf, seq_copy, slab_val), daemon=True).start()

full_tab = W.VBox([W.HBox([run_btn]), W.HBox([prog, eta_lbl]), msg_lbl, out_run,
                   W.HTML("<b>Diagnostics</b>"), W.HBox([diag_btn]), out_diag])
run_btn.on_click(run_full)

# ------------------ Assemble (2-pane) ------------------
tabs = W.Tab()
tabs.children = (W.VBox([param_holder]), preview_tab, full_tab)
tabs.set_title(0, "Filter Editor")
tabs.set_title(1, "Preview")
tabs.set_title(2, "Full Run / Logs")

header = W.Box([W.HTML("<h3 style='margin:6px 0;'>General Denoise</h3>")],
               layout=W.Layout(height="80px", align_items="center", padding="0 8px"))

try:
    app = W.AppLayout(
        header=header,
        left_sidebar=W.Box([W.VBox([left_panel])], layout=W.Layout(width="620px")),
        center=W.Box([tabs], layout=W.Layout(overflow="auto")),
        right_sidebar=None, footer=None,
        pane_widths=['620px','1fr','0px'], pane_heights=['80px','1fr','0px'],
    )
    display(app)
except Exception as e:
    # fallback if AppLayout not supported in this ipywidgets version
    display(W.VBox([
        header,
        W.HBox([
            W.Box([left_panel], layout=W.Layout(width="800px")),
            W.Box([tabs], layout=W.Layout(flex="1 1 auto"))
        ])
    ]))
    print("AppLayout fallback used due to:", repr(e))

free_bytes, total_bytes = B.gpu_mem_info()
print("Backend:", "GPU" if B.is_gpu() else "CPU", "| Free VRAM:", free_bytes, "of", total_bytes)
print("Algorithms (available):", ", ".join(_disp(k) for k in sorted(AVAIL)))


AppLayout(children=(Box(children=(HTML(value="<h3 style='margin:6px 0;'>General Denoise</h3>"),), layout=Layou…

Backend: GPU | Free VRAM: 13586399232 of 17094475776
Algorithms (available): Anisotropic Diffusion, Bilateral, Contrast-Dependent Outlier Removal, CLAHE, FFT, Gaussian, Guided, Kalman, K-NN, Laplacian of Gaussian, Mean, Median, Midrange, Non-local means, Unsharp, Wiener


# General Denoise — Filters, Ideas & Tuning Cheat-Sheet

| Filter (UI name) | Short key | Idea (what it does) | Typical key params (defaults) |
|---|---:|---|---|
| **Median (3D)** | `median3d` | Nonlinear rank filter; replaces each voxel with median in a local 3D window. Robust to salt/impulse noise; preserves edges better than mean. | `size=(3,3,3)` *(odd)* |
| **Mean (3D)** | `mean3d` | Linear box filter (uniform weights); fast baseline smoothing; blurs edges. | `size=(3,3,3)` |
| **Gaussian (3D)** | `gaussian3d` | Linear separable convolution with Gaussian kernel; smooth, edge-agnostic. | `sigma=(1,1,1)` |
| **Unsharp** | `unsharp3d` | Sharpen by adding back high-frequency detail: `x + amount*(x - Gσ(x))`, optional threshold to avoid boosting noise. | `sigma=(1,1,1)`, `amount=0.6`, `threshold=0.0` |
| **Laplacian of Gaussian** | `log3d` | Edge/structure enhancement by convolving with LoG; zero-crossings highlight edges. | `sigma=(1,1,1)`, `enhance=True` |
| **Bilateral (3D)** | `bilateral3d` | Edge-preserving smoothing: spatial Gaussian × range (intensity) Gaussian. Keeps edges while denoising flats. | `radius=(1,2,2)`, `sigma_s=(1,2,2)`, `sigma_r=0.1` |
| **Non-Local Means (3D)** | `nlm3d` | Patch-based averaging weighted by patch similarity; excels on fine, textured signals. | `patch_radius=(1,1,1)`, `search_radius=(2,6,6)`, `h≈0.6–1.0·σ` |
| **Contrast-Dependent Outlier Removal** | `cdor3d` | Suppress impulsive/speckle outliers using a contrast-aware threshold; preserves true edges. | `win=(3,5,5)`, `k_sigma=3.0`, `alpha=0.4`, advanced: `sigma_mode`, `contrast`, `replace` |
| **Anisotropic Diffusion** | `anisotropicdiff3d` | Perona–Malik diffusion: smooths inside regions, inhibits across edges via conductance. | `iters=50`, `k=0.05`, `lam=0.15`, `vox=(1,1,1)` |
| **Midrange** | `midrange3d` | Local `(min+max)/2`—robust to symmetric noise; can flatten contrast. | `win=(3,5,5)` |
| **Wiener (3D)** | `wiener3d` | Local mean/variance estimate; shrinks toward mean with noise variance compensation. | `win=(3,5,5)`, `sigma_n=None` *(auto if None)* |
| **Guided (3D)** | `guided3d` | Edge-aware linear model: within a window, fit `p ≈ a·I + b` and output `q = mean(a)·I + mean(b)`. Preserves edges guided by `I` (here `I=p`). | `r=(1,2,2)`, `eps=1e-3` |
| **Kalman (Z-recursive)** | `kalman3d` | 1D Kalman along Z (per-pixel) for temporal/axial smoothing; adapts to signal changes. | `R=0.01`, `Q=1e-4`, `init_var=1.0` |
| **k-Nearest Neighbour** | `knn3d` | In a local window, pick K voxels closest in intensity to center; average (optionally weighted). | `radius=(2,2,2)`, `K=9`, `metric={"l1","l2"}` |
| **FFT (2D/3D)** | `fft3d` | Frequency-domain filtering with masks (low/high/band) of type Ideal/Butterworth/Gaussian; careful with ringing/overlap. | `mode={"3d","2d"}`, `filter_kind`, `pass_type`, `order`, Nyquist/abs cutoffs |
| **CLAHE (per-slice)** | `clahe3d` | Local histogram equalization with clip limit; boosts local contrast while limiting noise amplification. | `clip_limit=0.01`, `tile=(8,8)`, `method={"auto","opencv","skimage"}` |

> **2D usage:** for filters with Z parameters, set Z-size/radius to 1 (or `sigma_z=0`) to operate per-slice.

---

# 3D Denoising & Filtering – Practical Guide for New Users

This note explains **what each filter does**, its **parameters**, **how to tune them**, and **when to choose which filter**.

It also covers **troubleshooting** and **memory-safe “SLAB streaming”** for large 3D volumes.

---

## 0. Notation & Basic Assumptions

We assume a 3D volume

$$
u(z,y,x) \in \mathbb{R}
$$

- \(z\) – slice index (depth)  
- \(y\) – row  
- \(x\) – column  

Most filters work in **normalized intensity**:

- Input working volume: `float32` scaled roughly to \([0, 1]\).
- The pipeline takes care of converting from `uint8` / `uint16` → `[0,1]` and back.

We will write:

- \(f(z,y,x)\) – noisy input  
- \(\hat{f}(z,y,x)\) – filtered output  

---

## 1. Local Neighborhood Filters (Simple & Fast)

### 1.1 Mean / Gaussian Smoothing (`mean3d`, `gaussian3d`)

**Purpose**

- Reduce random noise by averaging neighboring voxels.
- Gaussian is a *weighted* mean that preserves more details.

**Definition**

For a kernel \(w(\Delta z, \Delta y, \Delta x)\) that sums to 1:

$$
\hat{f}(z,y,x) \;=\; \sum_{\Delta z,\Delta y,\Delta x}
w(\Delta z,\Delta y,\Delta x)\, f(z+\Delta z, y+\Delta y, x+\Delta x)
$$

- Mean filter: \(w = \tfrac{1}{N}\) inside a window, 0 outside.  
- Gaussian filter:

$$
w(\Delta z,\Delta y,\Delta x) \propto 
\exp\!\left( -\frac{\Delta z^2}{2\sigma_z^2}
          -\frac{\Delta y^2}{2\sigma_y^2}
          -\frac{\Delta x^2}{2\sigma_x^2} \right)
$$

**Key parameters**

- `size` (mean) or `sigma` (gaussian) – spatial scale of smoothing
- `iterations` – how many times to apply the filter

**How to tune**

- Start with **small** kernels: e.g. `size=(3,3,3)` or `sigma=(1,1,1)`.
- Increase `sigma` or `size` if noise is strong, but watch for **blurred edges**.
- Use `iterations>1` instead of huge `sigma` to gradually increase smoothing.

**When to pick**

- Baseline smoothing.
- Pre-cleaning before more advanced filters.
- When performance matters more than perfect edge preservation.

---

### 1.2 Median & Midrange Filters (`median3d`, `midrange3d`)

**Purpose**

- Remove **impulse / salt-and-pepper noise** and outliers while preserving edges better than mean.

**Definition – Median**

In a local window \(\mathcal{N}(z,y,x)\):

$$
\hat{f}(z,y,x) \;=\; \operatorname{median}\{ f(z',y',x') : (z',y',x') \in \mathcal{N}(z,y,x) \}
$$

**Definition – Midrange**

Let \(m_{\min} = \min_{\mathcal{N}} f\), \(m_{\max} = \max_{\mathcal{N}} f\). Then

$$
\hat{f}(z,y,x) \;=\; \frac{m_{\min} + m_{\max}}{2}
$$

**Key parameters**

- `size` / `win` – window size (odd integers, e.g. `(3,3,3)`)
- `iterations` – how many passes

**How to tune**

- Use small windows `(3,3,3)` for fine structures.
- Increase window only when noise spikes are large.
- Too big windows → shape deformation and loss of fine details.

**When to pick**

- Data with **sparse spikes** or **hot pixels**.
- As a pre-filter before more sophisticated methods.

---

## 2. Edge-Preserving Local Filters

### 2.1 Bilateral Filter (`bilateral3d`)

**Purpose**

- Smooth noise **within regions** but preserve edges by reducing smoothing across intensity jumps.

**Definition (simplified)**

$$
\hat{f}(p) \;=\;
\frac{
\displaystyle \sum_{q \in \mathcal{N}(p)} 
    \exp\!\left(-\frac{\|p-q\|^2}{2\sigma_s^2}\right)
    \exp\!\left(-\frac{(f(p)-f(q))^2}{2\sigma_r^2}\right)
    f(q)
}{
\displaystyle \sum_{q \in \mathcal{N}(p)} 
    \exp\!\left(-\frac{\|p-q\|^2}{2\sigma_s^2}\right)
    \exp\!\left(-\frac{(f(p)-f(q))^2}{2\sigma_r^2}\right)
}
$$

- \(p=(z,y,x)\), \(q\) is a neighbor.
- \(\sigma_s\) – spatial scale.
- \(\sigma_r\) – range (intensity) scale.

**Key parameters**

- `radius=(rz,ry,rx)` – spatial window radius.
- `sigma_s` – controls spatial Gaussian.
- `sigma_r` – controls intensity similarity.

**How to tune**

- Start with small radius (e.g. `(1,3,3)`).
- `sigma_r`:
  - Too small → preserves edges but may leave noise.
  - Too large → behaves like Gaussian blur.
- Rule of thumb: `sigma_r` ≈ 1–3× noise standard deviation (in `[0,1]` scale).

**When to pick**

- You want **smooth, yet edge-preserved** images.
- Data with clear edges and moderate noise.

---

### 2.2 Guided Filter (`guided3d`)

**Purpose**

- Edge-preserving smoothing using a **local linear model** guided by an image (often the same image).

**Model**

Assume locally:

$$
q(z,y,x) \;=\; a_k\,I(z,y,x) + b_k, \quad \forall (z,y,x) \in \text{window }k
$$

- \(I\) – guidance image (often \(f\) itself).
- \(q\) – output.
- \(a_k,b_k\) – linear coefficients in window \(k\).

The solution involves local means and variances:

$$
a_k \;=\; \frac{\mathrm{cov}(I,p)}{\sigma_I^2 + \varepsilon}, 
\qquad
b_k \;=\; \mu_p - a_k \mu_I
$$

Then \(q\) is obtained by averaging overlapping windows.

**Key parameters**

- `r=(rz,ry,rx)` – window radius.
- `eps` – regularization (small positive number).
- `guidance` – optional separate guidance volume.

**How to tune**

- Small `r` → local detail preservation, less smoothing.
- Larger `r` → more smoothing, risk of flattening.
- `eps`:
  - Very small → follows edges strongly, can amplify noise in flat regions.
  - Bigger `eps` → smoother result.

**When to pick**

- Need edge-preserving smoothing with **less halo/artifact** than bilateral.
- Want to use another image as a guidance (e.g., structural channel).

---

## 3. Variational / PDE-based Filters

### 3.1 Anisotropic Diffusion (`anisotropicdiff3d`)

**Purpose**

- Diffuse (smooth) **within** homogeneous regions, **suppress** diffusion across edges.

**Perona–Malik diffusion**

$$
\frac{\partial u}{\partial t} 
= \nabla \cdot \!\big( c(|\nabla u|)\, \nabla u \big)
$$

with conductivity \(c(\cdot)\), e.g.

- Exponential: 
  $$
  c(s) \;=\; \exp\!\left(-\left(\frac{s}{k}\right)^2\right)
  $$
- Reciprocal: 
  $$
  c(s) \;=\; \frac{1}{1 + (s/k)^2}
  $$

Discretized with time step \(\lambda\) and number of iterations.

**Key parameters**

- `iters` – number of inner diffusion steps.
- `k` – edge threshold; smaller `k` preserves edges more.
- `lam` – time step size (controls stability & speed).
- `func` – type of conductivity (`"exp"` or `"lorentz"`).
- `vox` – voxel spacing `(dz,dy,dx)`.

**How to tune**

- Start with `iters ≈ 20–50`, `k` around noise level (in `[0,1]` units).
- `lam` ~ 0.1 is often safe; too large → instability / artifacts.
- If edges wash out → increase `k` or reduce `iters`.
- If noise remains → increase `iters` gradually.

**When to pick**

- When you want **smooth yet sharp** regions and are okay with iterative/PDE computation.
- Works well for piecewise-smooth objects.

---

### 3.2 Total Variation (TV) – *not implemented yet but conceptually useful*

For context, classic TV denoising solves:

$$
\min_u \;\; \frac{1}{2}\|u - f\|_2^2 + \lambda \int \|\nabla u\|\, dV
$$

This produces **piecewise constant** regions and sharp edges.  
Your anisotropic diffusion is a related idea, but you might add TV later.

---

## 4. Nonlocal Filters (Use Similar Patches)

### 4.1 Non-Local Means (`nlm3d`)

**Purpose**

- Use **repeated patterns** (patch self-similarity) across the volume to reduce noise while preserving textures.

**Idea**

For each voxel \(p\), compare **patches** in a search window:

$$
w(p,q) \propto \exp\!\left( -\frac{\|P(p) - P(q)\|_2^2}{h^2} \right)
$$

$$
\hat{f}(p) \;=\; \frac{\sum_{q} w(p,q) f(q)}{\sum_{q} w(p,q)}
$$

- \(P(p)\) – patch around \(p\).
- \(h\) – filtering parameter (related to noise level).

**Key parameters**

- `patch_radius` – half size of patch; patch size = `(2*pr+1)^3`.
- `search_radius` – how far to look for similar patches.
- `h` – controls decay of weights with patch distance.
- `sigma` – optional noise estimate for bias correction.
- `use_box` – whether to approximate patch SSD with box filters.

**How to tune**

- Start with small patches: `(1,1,1)` or `(0,2,2)`.
- Search radius moderate: `(0,5,5)` or `(1,5,5)`; large radii are **expensive**.
- `h` ~ 1–2× estimated noise std (in `[0,1]`).
  - Too small → retains noise, very selective.
  - Too large → over-smooths.

**When to pick**

- Textured data or repeated structures.
- When you want **excellent denoising quality** and can afford more computation.

---

### 4.2 KNN Filter (`knn3d`)

**Purpose**

- For each voxel, find the **K nearest neighbors** (similar values in a spatial radius) and average them.

**Simplified idea**

- For each voxel \(p\), within a search window, measure distances:

$$
d(p,q) \;=\; |f(p) - f(q)|
$$

- Choose **K smallest** distances, compute

$$
\hat{f}(p) \;=\; \frac{\sum_{q \in \mathcal{K}(p)} w_{pq} f(q)}{\sum_{q \in \mathcal{K}(p)} w_{pq}}
$$

with optional weights \(w_{pq}\) based on distance.

**Key parameters**

- `radius` – spatial search radius.
- `K` – number of neighbors.
- `metric` – typically `"l2"` (Euclidean).
- `weighting` – how to weight neighbors (uniform vs distance-based).
- `w_sigma`, `w_eps` – weighting shape parameters.
- `chunk`, `tile` – GPU chunking and Y/X tiling (performance).

**How to tune**

- `radius`: small for local smoothing, larger for stronger nonlocal effect.
- `K`:  
  - small K → preserve more detail, less smoothing.  
  - larger K → more robust, but risk of oversmoothing.
- Use presets; KNN is easy to overshoot with huge radii/K.

**When to pick**

- When NLM is too slow but you still want some **nonlocal behavior**.
- As a simpler alternative to NLM.

---

## 5. Frequency-Domain Filtering (`fft3d`)

**Purpose**

- Filter out noise or unwanted structures based on **spatial frequency** (texture scale).

**Definition**

1. Compute FFT:

$$
F(\xi_z,\xi_y,\xi_x) \;=\; \mathcal{F}\{ f(z,y,x) \}
$$

2. Apply mask \(M(\xi)\):

$$
G(\xi_z,\xi_y,\xi_x) \;=\; M(\xi_z,\xi_y,\xi_x) \cdot F(\xi_z,\xi_y,\xi_x)
$$

3. Inverse FFT:

$$
\hat{f} \;=\; \mathcal{F}^{-1}\{ G \}
$$

**Typical masks**

- **Ideal** (hard cutoff):

$$
M(\|\xi\|) \;=\; 
\begin{cases}
1 & \text{if } f_{\text{lo}} \le \|\xi\| \le f_{\text{hi}} \\
0 & \text{otherwise}
\end{cases}
$$

- **Butterworth** (smooth):

$$
M(\|\xi\|) \;=\; \frac{1}{1 + \left(\frac{\|\xi\|}{f_c}\right)^{2n}}
$$

- **Gaussian**:

$$
M(\|\xi\|) \;=\; \exp\!\left( -\left(\frac{\|\xi\|}{f_c}\right)^2 \right)
$$

**Key parameters**

- `mode` – `"3d"` or `"2d"` (per-slice FFT).
- `filter_kind` – `"ideal"`, `"butter"`, `"gauss"`.
- `pass_type` – `"low"`, `"high"`, `"band"`.
- `order` – Butterworth order (steepness).
- `spacing` – voxel spacing `(dz,dy,dx)` for frequency axis.
- Cutoffs:
  - `use_nyquist` + `f_lo_nyq`, `f_hi_nyq` (fractions of Nyquist)
  - or `f_lo_abs`, `f_hi_abs` (absolute frequency)

**How to tune**

- For **denoising**, a **low-pass** filter:
  - Cut high frequencies above the scale of noise.
- For **edge enhancement**, a **high-pass** or “unsharp” combination.
- Use Butterworth or Gaussian to avoid ringing from sharp ideal masks.
- Start with mild cutoffs (e.g., `f_hi_nyq=0.25`) and adjust visually.

**When to pick**

- Data with noise dominated in a certain **frequency band**.
- Want to suppress periodic patterns or emphasize structures at certain scales.

---

## 6. Statistical & Model-Based Filters

### 6.1 Wiener Filter (`wiener3d`)

**Purpose**

- Local, statistically motivated filter assuming additive Gaussian noise and locally stationary signal.

**Local Wiener formula**

Let within a window:

- \(\mu\) – local mean  
- \(\sigma^2\) – local variance  
- \(\sigma_n^2\) – noise variance

Then

$$
\hat{f} \;=\; \mu + \frac{\sigma^2 - \sigma_n^2}{\sigma^2} \,(f - \mu)
$$

(clamped so \(\sigma^2 \ge \sigma_n^2\)).

**Key parameters**

- `win` – window size.
- `sigma_n` – noise standard deviation (in `[0,1]`); if None, estimated from data.

**How to tune**

- Window: moderately larger than features of interest.
- `sigma_n`:
  - If known from acquisition, set explicitly.
  - If estimated, verify visually; overestimating → oversmoothing.

**When to pick**

- When noise is close to **white Gaussian** and you want something more adaptive than a simple blur.

---

### 6.2 Kalman Filter Along Z (`kalman3d`)

**Purpose**

- Temporal (or slice-wise) smoothing based on a **1D Kalman filter per voxel**.

**Simplified 1D model**

Prediction:

$$
\hat{x}_{k|k-1} \;=\; \hat{x}_{k-1|k-1},
\qquad
P_{k|k-1} \;=\; P_{k-1|k-1} + Q
$$

Update:

$$
K_k \;=\; \frac{P_{k|k-1}}{P_{k|k-1} + R},
\qquad
\hat{x}_{k|k} \;=\; \hat{x}_{k|k-1} + K_k \,(z_k - \hat{x}_{k|k-1}),
\qquad
P_{k|k} \;=\; (1 - K_k)\, P_{k|k-1}
$$

- \(Q\) – process noise variance.
- \(R\) – measurement noise variance.

**Key parameters**

- `Q` – how much you expect the true signal to change between slices.
- `R` – noise variance in observations.
- `init_var` – initial state variance.
- `iterations` – number of forward passes.

**How to tune**

- If slices are very similar (slowly varying):
  - Use **small Q**, larger smoothing.
- If signal changes rapidly along Z:
  - Larger Q to avoid lag / oversmoothing.
- R relates to noise level; larger R → more trust in model, less in noisy data.

**When to pick**

- 3D volumes where Z behaves like **time** (e.g., time series) or smoothly varying depth.
- When you want temporally stable result with parametric control.

---

### 6.3 CLAHE (`clahe3d`)

**Purpose**

- **Contrast Limited Adaptive Histogram Equalization**: enhance local contrast, especially in dark or bright regions.

**Idea (2D slice)**

- For each tile, compute histogram + cumulative distribution.
- Limit (clip) high histogram bins to avoid noise amplification.
- Map intensities via local cumulative distribution → improve local contrast.

**Key parameters**

- `clip_limit` – how much to clip histogram peaks (prevents noise over-amplification).
- `nbins` – number of histogram bins.
- `tile` – tile size per slice.
- `z_mode` – `"per_slice"` (current) vs potential future `"3d"`.

**How to tune**

- Lower `clip_limit` → more aggressive clipping, less noise amplification, but smaller contrast enhancement.
- Larger tiles → more global behavior; smaller tiles → more local enhancement (but risk of patchiness).
- CLAHE is **not a denoiser**; often combine with a denoising step before or after.

**When to pick**

- Low-contrast images where structures are hard to see.
- As a final step after denoising to improve visibility.

---

## 7. Unsharp & LoG Filters

### 7.1 Laplacian of Gaussian (`log3d`)

**Purpose**

- Edge detection / enhancement by highlighting areas of rapid intensity change.

**Definition**

Apply Gaussian smoothing followed by Laplacian:

$$
\hat{f} \;=\; \nabla^2 \!\big( G_\sigma \ast f \big)
$$

Where \(G_\sigma\) is Gaussian, \(\nabla^2\) is Laplacian.

**Key parameters**

- `sigma` – scale of structures emphasized.
- `enhance` – if True, combine with original to sharpen.

**How to tune**

- Small `sigma` → enhances fine details; more sensitive to noise.
- Larger `sigma` → enhances coarser edges.

---

### 7.2 Unsharp Masking (`unsharp3d`)

**Purpose**

- Sharpen image by adding a scaled edge component.

**Definition**

$$
g \;=\; f - G_\sigma \ast f \quad (\text{detail})
$$

$$
\hat{f} \;=\; f + \alpha\, g
$$

- \(G_\sigma\) – Gaussian.
- \(\alpha\) – amount of sharpening.

**Key parameters**

- `sigma` – blur scale for detail extraction.
- `amount` – \(\alpha\), how strong the sharpening is.
- `threshold` – ignore small differences to avoid sharpening noise.

**How to tune**

- Start with small `amount` (e.g. 0.3–0.5).
- Use threshold to suppress sharpening in uniform noisy regions.
- Combine after denoising for best results.

---

## 8. How to Choose a Filter (Heuristics)

- **Quick, simple denoising, low noise:**
  - `gaussian3d`, `mean3d`, `median3d`, `wiener3d`.
- **Impulse / salt-and-pepper noise:**
  - `median3d`, `midrange3d`, `cdor3d`.
- **Strong noise but want good detail:**
  - `nlm3d`, `knn3d`, `anisotropicdiff3d`, `guided3d`, `bilateral3d`.
- **Frequency-structured noise or periodic artifacts:**
  - `fft3d` (low/high/band-pass).
- **Contrast enhancement (not pure denoising):**
  - `clahe3d`, `unsharp3d`, `log3d`.
- **Smooth variation along one dimension (Z/time):**
  - `kalman3d` + others.

A common workflow:

1. **Basic denoise** (e.g. Gaussian or Wiener).
2. **Edge-preserving refinement** (e.g. bilateral / guided / anisotropic / NLM).
3. **Contrast enhancement** (e.g. CLAHE, unsharp) if needed.

---

## 9. Troubleshooting & Performance

### 9.1 Artifacts & Quality Issues

- **Over-smoothing / loss of detail**
  - Reduce kernel size (`sigma`, `size`, `win`).
  - Reduce `iters` / `iterations`.
  - Increase edge thresholds (`k` in anisotropic).
- **Residual noise**
  - Increase iterations gradually.
  - Slightly increase `sigma` / `h` / `clip_limit`.
  - Combine a simple filter (Gaussian) with a stronger one (NLM).
- **Halos / ringing (FFT filters)**
  - Avoid sharp “ideal” filters; use Butterworth or Gaussian.
  - Check that intensities and boundaries are well-behaved (no abrupt cut at volume border).
- **Grainy / patchy look (CLAHE, anisotropic with bad settings)**
  - Increase tile size or reduce `clip_limit`.
  - Reduce iterations.

### 9.2 Perf & Memory: SLAB Streaming (Full-volume, Memory-safe)

Large 3D volumes may not fit in GPU memory. The pipeline uses **SLAB streaming**:

- Split volume into slabs along Z: `[z0:z1]`.
- Add small **overlap** in Z to avoid boundary artifacts.
- Process each slab independently on GPU.
- Write processed slices back to disk.
- Repeat until full volume is processed.

**Why this helps**

- Memory usage scales with slab depth, not full volume size.
- Keeps GPU memory within safe limits, even for huge datasets.

**Possible failure points**

- **Out-of-Memory (OOM) on GPU**
  - Reduce slab depth (configure `slab_z`).
  - Lower search radii for NLM/KNN.
- **Runtime too slow**
  - Avoid huge search windows.
  - Use “Balanced” or “Fast” presets.
  - Prefer simpler filters (Gaussian, Wiener, guided) over NLM when exploring.

---

**Final advice:**  
Start with **simple filters** to understand noise level and structure, then layer more advanced methods (NLM, anisotropic diffusion, guided, FFT) only when necessary. Always inspect both **noise reduction** *and* **detail preservation** – good denoising is a balance, not an absolute.


# 3D Denoising & Filtering — Practical Guide

This note explains **what each filter does**, its **parameters**, **how to tune them**, and **when to use which filter**.  
It also covers **memory-safe “SLAB streaming”** for large 3D volumes.

---

## 0) Notation & Working Assumptions

Volume shape **(Z, Y, X)**, grayscale. Processing runs on `float32` in **[0,1]** (the pipeline converts from/to `uint8`/`uint16`/float automatically).

We denote noisy input \(f\) and filtered output \(\hat f\).

---

## 1) Local Neighborhood Filters (simple, fast)

### Mean (`mean3d`) and Gaussian (`gaussian3d`)

**Idea.** Average neighbors; Gaussian is a weighted average.

$$
\hat f(p)=\sum_{q\in\mathcal N(p)} w(p-q)\,f(q)
$$

Gaussian weights:

$$
w_{\text{gauss}}(\Delta z,\Delta y,\Delta x)\ \propto\
\exp\!\Big(-\frac{\Delta z^2}{2\sigma_z^2}-\frac{\Delta y^2}{2\sigma_y^2}-\frac{\Delta x^2}{2\sigma_x^2}\Big)
$$

**Key params.** Mean: `size=(kz,ky,kx)` (odd recommended). Gaussian: `sigma=(σz,σy,σx)`. `iterations` repeats the filter.

---

### Median (`median3d`) & Midrange (`midrange3d`)

**Median:**
$$
\hat f(p)=\operatorname{median}\{\,f(q):q\in\mathcal N(p)\,\}
$$

**Midrange:**
$$
\hat f(p)=\tfrac12\Big(\min_{q\in\mathcal N(p)} f(q)+\max_{q\in\mathcal N(p)} f(q)\Big)
$$

**Key params.** `size` / `win` (odd), `iterations`.

---

## 2) Edge-Preserving Local Filters

### Bilateral (`bilateral3d`)

**Idea.** Spatial + range weighting; smooth within regions, preserve edges.

$$
\hat f(p)=\frac{\displaystyle \sum_{q\in\mathcal N(p)} 
e^{-\frac{\|p-q\|^2}{2\sigma_s^2}}\,
e^{-\frac{(f(p)-f(q))^2}{2\sigma_r^2}}\,
f(q)}
{\displaystyle \sum_{q\in\mathcal N(p)} 
e^{-\frac{\|p-q\|^2}{2\sigma_s^2}}\,
e^{-\frac{(f(p)-f(q))^2}{2\sigma_r^2}}}
$$

**Key params.** `radius=(rz,ry,rx)`, `sigma_s=(σz,σy,σx)`, `sigma_r`.

---

### Guided (`guided3d`)

**Idea.** Local linear model with the input as guidance:

$$
q = a_k\,I + b_k,\qquad 
a_k=\frac{\operatorname{cov}(I,p)}{\sigma_I^2+\varepsilon},\ \ 
b_k=\mu_p-a_k\mu_I
$$

**Key params.** `r=(rz,ry,rx)`, `eps` (in \([0,1]^2\)).

---

## 3) Variational / PDE

### Anisotropic Diffusion (`anisotropicdiff3d`)

**Perona–Malik:**
$$
\frac{\partial u}{\partial t}=\nabla\cdot\!\Big(c(|\nabla u|)\,\nabla u\Big),
\qquad
c(s)=\exp\!\big(-(s/k)^2\big)\ \text{or}\ \ (1+(s/k)^2)^{-1}
$$

**Key params.** `iters`, `k`, `lam`, `func∈{exp,lorentz}`, `vox=(dz,dy,dx)`.

---

## 4) Nonlocal

### Non-Local Means (`nlm3d`)

Weights by patch similarity:

$$
w(p,q)\ \propto\ \exp\!\Big(-\tfrac{\|P(p)-P(q)\|_2^2}{h^2}\Big),\qquad
\hat f(p)=\frac{\sum_q w(p,q)\,f(q)}{\sum_q w(p,q)}
$$

**Key params.** `patch_radius`, `search_radius`, `h`, optional `sigma`, `use_box`.

---

### K-NN (`knn3d`)

Pick K nearest intensities in a radius and combine:

$$
\hat f(p)=\frac{\sum_{q\in\mathcal K(p)} w_{pq}\,f(q)}{\sum_{q\in\mathcal K(p)} w_{pq}}
$$

**Key params.** `radius`, `K`, `metric∈{l1,l2}`, `weighting∈{inverse,inverse_sq,gaussian,uniform}`, `w_sigma`, `w_eps`.

---

## 5) Frequency-Domain (`fft3d`)

FFT → mask → iFFT:

$$
F=\mathcal F\{f\},\quad G(\xi)=M(\xi)\,F(\xi),\quad \hat f=\mathcal F^{-1}\{G\}
$$

**Masks (examples).**

Ideal (band):
$$
M(r)=\begin{cases}
1,& f_{\text{lo}}\le r\le f_{\text{hi}}\\
0,& \text{otherwise}
\end{cases}
$$

Butterworth (low-pass):
$$
M(r)=\frac{1}{1+\big(\tfrac{r}{f_c}\big)^{2n}}
$$

Gaussian (low-pass):
$$
M(r)=\exp\!\Big(-\big(\tfrac{r}{f_c}\big)^2\Big)
$$

**Key params.** `mode∈{3d,2d}`, `filter_kind∈{ideal,butter,gauss}`, `pass_type∈{low,high,band}`, `order` (Butterworth only), `spacing=(dz,dy,dx)`, and Nyquist fractions (`use_nyquist` + `f_*_nyq`) **or** absolute (`f_*_abs`).

---

## 6) Statistical / Model-based

### Wiener (`wiener3d`)

Local shrinkage:
$$
\hat f=\mu+\frac{\max(\sigma^2-\sigma_n^2,0)}{\max(\sigma^2,\varepsilon)}\,(f-\mu)
$$

**Key params.** `win` (odd), `sigma_n` = noise **std** in `[0,1]`.

---

### Kalman along Z (`kalman3d`)

Prediction–update per voxel:
$$
\hat x_{k|k-1}=\hat x_{k-1|k-1},\quad
P_{k|k-1}=P_{k-1|k-1}+Q
$$

$$
K_k=\frac{P_{k|k-1}}{P_{k|k-1}+R},\quad
\hat x_{k|k}=\hat x_{k|k-1}+K_k\big(z_k-\hat x_{k|k-1}\big),\quad
P_{k|k}=(1-K_k)P_{k|k-1}
$$

**Key params.** `Q`, `R`, `init_var`, `iterations`.

---

## 7) Contrast Enhancement

### CLAHE (`clahe3d`)

Adaptive histogram equalization with clipping.  
**Key params.** `clip_limit`, `nbins`, `tile=(Ty,Tx)`, `z_mode∈{per_slice,3d}`, `method∈{auto,gpu,skimage}`.  
`3d` uses cuCIM GPU when available; `per_slice` uses skimage on each slice.

---

## 8) Laplacian of Gaussian  & Unsharp

**Laplacian of Gaussian (`log3d`):**
$$
\hat f = \nabla^2\!\big(G_\sigma * f\big)
$$

**Unsharp (`unsharp3d`):**
$$
g=f-G_\sigma * f,\qquad \hat f=f+\alpha\,g
$$

---

## 9) SLAB Streaming (memory-safe full volume)

Process core slab \([z_0:z_1)\) with a Z-overlap for context. Read extended \([z_s:z_e)\), filter, crop back to core, write, advance.

**Overlap rules (thumb):**
- Median: \( \lfloor k_z/2 \rfloor \)
- NLM: \( r_z + s_z \)
- Gaussian: \( \lceil 3\sigma_z \rceil \)
- FFT-3D: larger for sharper masks; 2D mode needs **no Z overlap**.


# FFT Filters (3D / 2D) — Quick Reference

## 1) What these filters do
FFT-domain filtering multiplies the Fourier transform of your volume by a **frequency mask**:
\[$
\hat{u}(\mathbf{f}) = \hat{v}(\mathbf{f}) \cdot M(\mathbf{f})
\quad\Rightarrow\quad
u = v * h$
\]
where \(M\) is the mask (low/high/band pass), and \(h\) is its spatial impulse response. **Sharper masks ⇢ longer spatial tails** (more ringing; need larger overlap when tiling).

---

## 2) Coordinate system & units
- Volume shape: **(Z, Y, X)**.  
- Voxel spacing \( $\Delta = (d_z, d_y, d_x)$ \) in physical units (e.g., µm).  
- Frequency grids (cycles per unit): `fftfreq(N, d)` gives \($[-1/(2d), 1/(2d]$)\).  
- **Nyquist per-axis**: \($f_{N,z}=1/(2d_z)$\), \($f_{N,y}=1/(2d_y$)\), \($f_{N,x}=1/(2d_x)$\).  
- **Radial frequency**: \($ f_r = \sqrt{f_z^2 + f_y^2 + f_x^2}$ \).  
- If spacing is unknown, use \($\Delta=(1,1,1)$\) (cycles per pixel/voxel).

---

## 3) Modes
- **3D mode**: true 3D FFT over (Z,Y,X). Best fidelity; highest VRAM.  
- **2D mode**: per-slice FFT over (Y,X). Lower VRAM; no Z coupling; easiest to stream.

---

## 4) Pass types & mask families
**Pass types**
- **Low-pass**: keep large structures; cutoff \($f_{\text{hi}}$\).
- **High-pass**: keep fine detail; cutoff \($f_{\text{lo}}$\).
- **Band-pass**: keep \( $f_{\text{lo}} \le f_r \le f_{\text{hi}}$ \).

**Mask families (choose one)**
- **Ideal (brick-wall)**: hard cutoff. Max ringing (Gibbs), infinite support → avoid tiling if possible.
- **Butterworth (order \(n\))**: smooth roll-off. Higher \(n\) = steeper edge, longer tails (needs more overlap).
- **Gaussian**: smoothest; fastest tail decay; most tile-friendly.

---

## 5) Choosing cutoffs
Two equivalent ways:

**A) Nyquist fractions (no physical units):**
- Pick \( $\eta \in (0,1]$ \).  
- Low-pass: \( $f_{\text{hi}} = \eta \cdot \min(f_{N,z}, f_{N,y}, f_{N,x})$ \).  
- High-pass: \( $f_{\text{lo}} = \eta \cdot \min(\cdot)$ \).

**B) Physical wavelengths (recommended if spacing known):**
- To **keep** features ≥ \($L$\): set low-pass \($ f_{\text{hi}} \approx 1/L$\).  
- To **remove** trends ≥ \($L$\): set high-pass \( $f_{\text{lo}} \approx 1/L $\).  
- For **band-pass** targeting sizes in \($[L_{\min}, L_{\max}]$\):  
  \( $f_{\text{lo}} \approx 1/L_{\max}, \; f_{\text{hi}} \approx 1/L_{\min}$ \).

**Good starting points**
- Denoise fine speckle: Butterworth **low-pass**, order 2, \($ \eta \approx 0.20\!-\!0.30$ \).  
- Background flattening: Butterworth **high-pass**, order 2, \( $\eta \approx 0.05\!-\!0.10$ \).  
- Structure banding: Butterworth **band-pass**, order 2, set \($[f_{\text{lo}}, f_{\text{hi}}]$\) from size range.

---

## 6) Slabbed full-volume processing (to reduce memory)
Filtering in frequency equals **convolution** with kernel \(h\). When streaming in **Z-slabs**, add **Z-overlap** large enough to capture the effective support of \(h\).

**Rule of thumb for Z-overlap**
1. Pick effective cutoff \( $f_{\text{eff}}$ \):
   - Low-pass: \( $f_{\text{eff}} = f_{\text{hi}} $\)
   - High-pass: \( $f_{\text{eff}} = f_{\text{lo}}$ \)
   - Band-pass: \($ f_{\text{eff}} = \min(f_{\text{lo}}, f_{\text{hi}}) $\)
2. Choose \($k$\) by mask smoothness:
   - Gaussian: \($k \approx 3$\)
   - Butterworth (order 2–4): \($k \approx 5\!-\!8$\)  (higher order ⇒ bigger \($k$\))
   - Ideal: **avoid slab tiling**; if forced, very large \($k$\) (often impractical)
3. Convert to overlap in **slices**:
\[$
L \approx \frac{k}{f_{\text{eff}}} \quad\Rightarrow\quad
\text{ovl}_z = \left\lceil \frac{L}{d_z} \right\rceil$
\]
Clamp to a sane range (e.g., 2…128). Increase if you see seams.

**Slab algorithm (Z streaming)**
- Core window: `[z0:z1)` of length **SLAB** (slices to output this pass).  
- Read extended: `[zs:ze)` with `$ zs=z0-ovl_z $`, `$ze=z1+ovl_z $` clipped to `[0,Z)`.  
- FFT-filter the **extended** slab (3D FFT).  
- Crop back to core and save.  
- Advance `z0 ← z1` and repeat.

> For **2D mode**, $ovl\_z$ = 0 (no Z context), so slabs can be large.

---

## 7) Practical notes & pitfalls
- **Ringing**: Sharper masks (Ideal, high-order Butterworth) ⇒ halos near edges & seams. Prefer Gaussian or order-2 Butterworth for microscopy-like data.  
- **Padding**: FFT assumes periodic boundaries. Mirror-padding before FFT can reduce edge ringing if you process the **entire** volume; for slabs we rely on overlap.  
- **Memory**: 3D FFT needs ~2–3× slab size in VRAM (complex spectra + temporaries). Reduce **SLAB** if you hit OOM.  
- **2D vs 3D**: 2D is faster/safer; use when through-plane correlations are weak or Z anisotropy is high.  
- **Evaluation**: Use your existing `quality_metrics_3d` (noise reduction in flats, edge retention on gradients) and visual checks on 2–3 representative slices.

---

## 8) Parameters (summary)
- `FILTER_KIND`: `"ideal" | "butter" | "gauss"`  
- `PASS_TYPE`: `"low" | "high" | "band"`  
- `ORDER_BUTTER`: 1–4 (steepness; 2 is a good default)  
- `SPACING=(d_z,d_y,d_x)`: voxel size (set `(1,1,1)` if unknown)  
- Cutoffs: **either** Nyquist fractions (`F_*_NYQ`) **or** absolute (`F_*_ABS`, cycles/unit)  
- `FFT_MODE`: `"3d"` (preferred if VRAM allows) or `"2d"`  
- `SLAB`: core slices per pass (e.g., 64)  
- `ovl_z`: computed from \(k, f_{\text{eff}}, d_z\) (see §6)

---

## 9) cuFFT dependency (CuPy)
- `cp.fft.*` requires **cuFFT** (e.g., `libcufft.so.11` for CUDA-12).  
- Minimal check (avoid `cp.random`, which needs cuRAND):
  ```python
  import cupy as cp, numpy as np
  x = cp.asarray(np.random.rand(8,8,8).astype(np.float32))  # NumPy RNG -> GPU
  ok = float(cp.abs(cp.fft.ifftn(cp.fft.fftn(x)).real - x).max()) < 1e-5
  print("cuFFT OK:", ok)
